In [6]:
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)

import numpy as np
import pandas as pd
from os import listdir
from os.path import isfile, join
from mhcnames import normalize_allele_name
import os
import time

import logging
import sys
from scipy.version import full_version as spversion
from config import  LOG_PATH

METHOD = "hmm_pomegranate"


# 1. Read data

In [8]:
# DATA_PATH = 'D:/Google Drive/Work/ITMO/data/' # windows
DATA_PATH = '/home/denklewer/Data/length_dependent_hmm/' #ubuntu



In [9]:
onlyfiles = [f for f in listdir(DATA_PATH) if isfile(join(DATA_PATH, f))]
ALLELES = []
per_allele_train_df = dict()
for file_name in onlyfiles:
    allele_name = file_name.split("_", 1)[1].split('.')[0]
    allele_name = normalize_allele_name(allele_name)
    per_allele_train_df[allele_name] =  pd.read_csv(DATA_PATH + file_name,sep=";")
    ALLELES.append(allele_name)
ALLELES

['HLA-A*02:01']

# 3. Train HMMS

In [10]:
#leave only binders
def getBindersPeptidesOnly(df):
    result_peptides = df[df['binder'] == 1]['peptide'].values
    return result_peptides
def getNonBindersPeptidesOnly(df):
    result_peptides = df[df['binder'] == 0]['peptide'].values
    return result_peptides

def make_per_length_dict(df, binder_flag):
    lengths = set(df['length'])
    per_length_dict = dict()
    for length in lengths:
        result_peptides = df[(df['binder'] == binder_flag) & (df['length'] == length)]['peptide'].values
        per_length_dict[length] = result_peptides
    return per_length_dict  

def addAuxColumns(per_allele_df):
    for allele_name, df in per_allele_df.items():
        df['length'] = df['peptide'].str.len()
        
per_allele_binders_train = dict()
per_allele_non_binders_train = dict()
per_allele_per_length_binders_train = dict()
per_allele_per_length_non_binders_train = dict()


for allele_name in ALLELES:
    df_train = per_allele_train_df[allele_name]
    per_allele_binders_train[allele_name] = getBindersPeptidesOnly(df_train)
    per_allele_non_binders_train[allele_name] = getNonBindersPeptidesOnly(df_train)
    per_allele_per_length_binders_train[allele_name] = make_per_length_dict(df_train, binder_flag=1)
    per_allele_per_length_non_binders_train[allele_name] = make_per_length_dict(df_train, binder_flag=0)

 

## Code for HMM

In [11]:
from pomegranate import *

model = HiddenMarkovModel()

In [12]:
model

TypeError: 'NoneType' object is not iterable

In [9]:
AminoAcidsZ = ghmm.Alphabet(ghmm.AminoAcids.listOfCharacters + ['Z'])

# evaluation code

def prepareHmmSequenceSets(peptides, stopEmission=True):
    if stopEmission:
        peptideList = [str(p) + 'Z' for p in peptides]
    else:
        peptideList = [str(p) for p in peptides]
    
    # need to split in blocks in case there are more than 1.5M peptides (ghmm limitation)
    ghmmMaxSequencesPerSet = 1500000
    lenPeptides = len(peptides)
    numFullBlocks = lenPeptides // ghmmMaxSequencesPerSet
    lenLastBlock = lenPeptides % ghmmMaxSequencesPerSet
    sequenceSets = []
    
    # full blocks (if any)
    for i in range(0, numFullBlocks):
        rangeStart = i * ghmmMaxSequencesPerSet
        rangeEnd = rangeStart + ghmmMaxSequencesPerSet
        sequenceBlock = ghmm.SequenceSet(
            AminoAcidsZ,
            [list(p) for p in peptideList[rangeStart:rangeEnd]]
        )
        sequenceSets.append(sequenceBlock)
    rangeStart = numFullBlocks * ghmmMaxSequencesPerSet
    rangeEnd = lenPeptides
    if (lenLastBlock > 0):
        sequenceBlock = ghmm.SequenceSet(
                AminoAcidsZ,
                [list(p) for p in peptideList[rangeStart:rangeEnd]]
            )
        sequenceSets.append(sequenceBlock)
    return sequenceSets

def predictHmmPeptidesRaw(hmmModelFile, sequenceSets):
    """Predict -10logp values using a given HMM model for binders and non-binders.
       Results will contain -inf values
    """
    hmm = ghmm.HMMOpen(hmmModelFile)

    return np.concatenate(
        [-10 * np.array(hmm.loglikelihoods(s)) for s in sequenceSets]
    )


def predict_with_ghmm(per_allele_df):
    per_allele_predictions = dict()
    for allele_name in ALLELES:
        print(allele_name)
        target_allele = TARGET_ALLELES[allele_name]
        df_val = per_allele_df[allele_name].copy()
        if target_allele in supported_alleles:
            sequenceSet = prepareHmmSequenceSets(df_val.peptide)
            print("Predict with model")
            print("Binders")
            df_val['HMMB'] = predictHmmPeptidesRaw('{}{}-splitwithdecoy-binders.xml'.format(MODEL_PATH, target_allele),sequenceSet)

            columnsHMM = ['HMMB']
            df_val.loc[:, columnsHMM] = df_val.loc[:, columnsHMM].div(df_val.length, axis=0)
            df_val = df_val.replace(np.inf, 10000.0)
        else:
            print("Allele {} is not supported, create dummy predictions".format(target_allele))
            df_val['HMMB'] = np.nan
        per_allele_predictions[allele_name] = df_val
        
    
    return per_allele_predictions

def update_scores_with_percentiles(per_allele_scores, per_allele_ref_scores, score_name):
    for allele_name in per_allele_scores.keys():
        df = per_allele_scores[allele_name].copy()
        ref_df =  per_allele_ref_scores[allele_name].copy()
        scores = df[score_name]
        ref_scores = ref_df[score_name]
        ref_scores = ref_scores.sort_values()
        pct_increment = 1./len(ref_scores)
        percentile_column_name = 'percentile_rank'
        percentile_ranks = (0.5 + 0.5*ref_scores.searchsorted(scores, side='left') + 0.5 * ref_scores.searchsorted(scores, side='right'))*pct_increment
        percentile_ranks = percentile_ranks*100
        per_allele_scores[allele_name][percentile_column_name] = percentile_ranks
    return per_allele_scores


# training code
def randomInitHMMwithSpecialStates(numStates, alphabet):
    """Initialize an HMM

    Create an HMM with a random state and dedicated stop state and emission
    numStates - number of hidden states in the HMM
    alphabet - the alphabet of emissions, of ghmm.Alphabet type,
    should contain X and Z emissions for start and stop
    """
    
    # all states equiprobable as initial, Baum-Welch will adjust
    pi = [1] * numStates
    
    # transition probabilities: all random except for the last state, which does not map into anything,
    # so ghmm will recognize it as the final state
    tT = np.random.rand(numStates, numStates)
    tT[numStates - 1] = np.array([0] * numStates)
    
    # emission probabilities: last state will be emitting only the dedicated stop state 'Z'
    # other options not possible:
    # - if we does not let the last state emit at all (silent state), ghmm.baumWelch
    #   will not work as it does not support silent states
    # - if we leave the random emission pattern for the last state, baumWelch complains
    #   that the state cannot be reached (And so the model is wrong)
    # so we need the dedicated stop state
    # Same for the start state
    tE = np.random.rand(numStates, len(alphabet))
    tE[numStates - 1] = np.zeros(len(AminoAcidsZ))
    tE[numStates - 1][AminoAcidsZ.internal('Z')] = 1
    for i in range(0, numStates - 1):
        tE[i][AminoAcidsZ.internal('Z')] = 0
    
    h = ghmm.HMMFromMatrices(alphabet, ghmm.DiscreteDistribution(alphabet), tT, tE, pi)
    h.normalize()
    return h

def trainHMMs(binders, nonbinders, numStatesBinders, numStatesNonbinders, llCutoff):
    
    logger.info(
        'Training HMM on peptides ({} B, {} NB)'.format(
            len(binders),
            len(nonbinders)
        )
    )
    
    # convert peptides to ghmm structure
    bindersTrainSS = ghmm.SequenceSet(
        AminoAcidsZ,
        [list(str(p) + 'Z') for p in binders.peptide])
    hmmB = randomInitHMMwithSpecialStates(numStatesBinders, AminoAcidsZ)
    tt0 = time.clock()
    hmmB.baumWelch(bindersTrainSS, loglikelihoodCutoff = llCutoff)
    tt1 = time.clock()
    logger.info('Binders model trained in {} s'.format(tt1 - tt0))
    
    nonbindersTrainSS = ghmm.SequenceSet(
        AminoAcidsZ,
        [list(str(p) + 'Z') for p in nonbinders.peptide])
    hmmN = randomInitHMMwithSpecialStates(numStatesNonbinders, AminoAcidsZ)
    tt0 = time.clock()
    hmmN.baumWelch(nonbindersTrainSS, loglikelihoodCutoff = llCutoff)
    tt1 = time.clock()
    logger.info('Non-binders model trained in {} s'.format(tt1 - tt0))
    
    return (hmmB, hmmN)





In [10]:
from sklearn.model_selection import KFold
for df in per_allele_train_df.values():
    print("max length {}".format(max(df.peptide.str.len())))
kf = KFold(n_splits=5)
llCutoff = 0.00005

max length 43


## Train 8 mers model

In [13]:
numStatesBinders = 12
numStatesNonbinders = 12

TARGET_LENGTH = 8

binders_array = per_allele_per_length_binders_train[allele_name][TARGET_LENGTH]
non_binders_array =  per_allele_per_length_non_binders_train[allele_name][TARGET_LENGTH]


In [17]:
i = 0
for train_indexes_b, test_indexes_b in kf.split(binders_array):
    i = i + 1
    logger.info("split {}".format(i))
    logger.info("train/test {}/{}".format(len(train_indexes_b), len(test_indexes_b)))

    current_train_b = binders_array[train_indexes_b]
    current_test_b = binders_array[test_indexes_b]
    current_train_nb = non_binders_array
    current_test_nb = non_binders_array

    df_binders = pd.DataFrame({"peptide":current_train_b})
    df_non_binders = pd.DataFrame({"peptide":current_train_nb})
    hmmB, hmmN = trainHMMs(df_binders,df_non_binders, numStatesBinders, numStatesNonbinders, llCutoff)
    hmmB.write('trained_models/{}-binders-length-{}-model-{}.xml'.format(allele_name, TARGET_LENGTH, i))
    hmmN.write('trained_models/{}-nonbinders-length-{}-model-{}.xml'.format(allele_name,TARGET_LENGTH, i))

GHMM <ipython-input-17-e96b8eaa05ad>:4 - split 1
GHMM <ipython-input-17-e96b8eaa05ad>:5 - train/test 2392/599
GHMM <ipython-input-9-2368c7169703>:127 - Training HMM on peptides (2392 B, 82 NB)
GHMM ghmm.py:148 - reestimate.c:ghmm_dmodel_baum_welch(853): Convergence after 46 steps
GHMM <ipython-input-9-2368c7169703>:139 - Binders model trained in 3.49847 s
GHMM ghmm.py:148 - reestimate.c:ghmm_dmodel_baum_welch(853): Convergence after 58 steps
GHMM <ipython-input-9-2368c7169703>:148 - Non-binders model trained in 0.152856 s
GHMM <ipython-input-17-e96b8eaa05ad>:4 - split 2
GHMM <ipython-input-17-e96b8eaa05ad>:5 - train/test 2393/598
GHMM <ipython-input-9-2368c7169703>:127 - Training HMM on peptides (2393 B, 82 NB)
GHMM ghmm.py:148 - reestimate.c:ghmm_dmodel_baum_welch(853): Convergence after 54 steps
GHMM <ipython-input-9-2368c7169703>:139 - Binders model trained in 4.073995 s
GHMM ghmm.py:148 - reestimate.c:ghmm_dmodel_baum_welch(853): Convergence after 50 steps
GHMM <ipython-input-9-236

[[3.0100881645190463e-137,
  1.1522678654697677e-134,
  5.299329329751864e-287,
  0.0,
  1.0,
  0.0,
  0.0,
  0.0,
  2.230271454541653e-140,
  0.0,
  0.0,
  4.480076227244209e-276],
 [0.0,
  1.293511465489699e-135,
  0.0,
  0.0,
  9.79867657300839e-288,
  0.0,
  0.0,
  0.0,
  1.0203431585123753e-138,
  0.0,
  0.0,
  0.9999999999999999],
 [1.0,
  3.376332635243949e-275,
  1.1620076430580922e-137,
  3.016752738831452e-283,
  2.4423335835185896e-136,
  0.0,
  0.0,
  1.6738385673610315e-295,
  3.5913291368294194e-280,
  0.0,
  0.0,
  0.0],
 [5.543492861077122e-135,
  0.0,
  1.0,
  1.6628324479999207e-136,
  2.002476929048295e-273,
  6.838492106757367e-299,
  0.0,
  6.231326887279533e-139,
  0.0,
  0.0,
  1.3752814357890583e-287,
  0.0],
 [1.3355987096597936e-283,
  0.7805786040452051,
  0.0,
  0.0,
  7.323943033817605e-148,
  0.0,
  0.0,
  0.0,
  0.21942139595478302,
  0.0,
  0.0,
  2.090969649916846e-137],
 [7.347327105429238e-288,
  0.0,
  4.79149279861091e-144,
  0.006297045583113342,
 